In [21]:
import torch


# Check if CUDA (NVIDIA GPU) is available
is_cuda = torch.cuda.is_available()
print(f"GPU Available (CUDA): {is_cuda}")

if is_cuda:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


GPU Available (CUDA): True
Number of GPUs: 1
GPU Name: Tesla T4


In [22]:
!pip install -q "pydantic-ai-slim[openai]" fastmcp nest_asyncio openai fastapi uvicorn requests pyjwt python-dotenv python-multipart python-weather aiohttp beautifulsoup4  huggingface_hub

In [23]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [25]:
# --- install Ollama and start it in the background ---
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os
# Use 0.0.0.0 to ensure the server is accessible within the Colab network
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

#use gemma 4 model for agentic tool calling
!ollama pull gemma4:e4b


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
# !nohup ollama serve > ollama.log 2>&1 &


In [26]:
!curl http://localhost:11434

Ollama is running

In [27]:
import nest_asyncio; nest_asyncio.apply()      # lets `await` work inside a notebook

from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

def ollama_model(name):
    # Ollama copies the OpenAI API shape, so we reuse the OpenAI client and swap the URL.
    return OpenAIChatModel(
        name,
        provider=OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama"),
    )

model = ollama_model("gemma4:e4b")
print("setup done")

setup done


#Session 1 - JSON RPC 2.0

In [28]:
request = {
    "jsonrpc": "2.0",
    id: 1,
    "method": "add",
    "params": {
        "a": 2,
        "b": 3
    }
}

Response Result or Error

In [29]:
ok  = {"jsonrpc": "2.0", "id": 1, "result": 5}
bad = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}

## The server: a lookup and a call

1.   List item
2.   List item




In [30]:
def add(a, b):
  return a + b
def shout(text):
  return text.upper() + "!"

FUNCTIONS = {"add": add, "shout": shout}      # name (text) -> real Python function


def handle(req):
    fn = FUNCTIONS.get(req["method"])                       # look up by NAME

    if fn is None:                                          # no such function -> error reply
        return {"jsonrpc": "2.0", "id": req["id"],
                "error": {"code": -32601, "message": "Method not found"}}

    result = fn(**req["params"])                            # <-- THE REMOTE CALL.
                                                            #     ** turns {"a":2,"b":3} into add(a=2, b=3)
    return {"jsonrpc": "2.0", "id": req["id"], "result": result}


print(handle({"jsonrpc": "2.0", "id": 1, "method": "add",   "params": {"a": 40, "b": 2}}))
print(handle({"jsonrpc": "2.0", "id": 2, "method": "shout", "params": {"text": "hello"}}))
print(handle({"jsonrpc": "2.0", "id": 3, "method": "fly",   "params": {}}))

{'jsonrpc': '2.0', 'id': 1, 'result': 42}
{'jsonrpc': '2.0', 'id': 2, 'result': 'HELLO!'}
{'jsonrpc': '2.0', 'id': 3, 'error': {'code': -32601, 'message': 'Method not found'}}


# SESSION 2 - MCP BY HAND

In [31]:
def add(a, b):
  return a + b
def shout(text):
  return text.upper() + "!"

FUNCTIONS = {"add": add, "shout": shout}      # name (text) -> real Python function


TOOLS = [
    {
        "name": "add",                                                   # matches FUNCTIONS["add"]
        "description": "Add two numbers together and return the sum.",   # the LLM reads this
        "inputSchema": {
            "type": "object",                        # arguments always arrive as a dict
            "properties": {"a": {"type": "number"},  # argument a -> a number
                           "b": {"type": "number"}}, # argument b -> a number
            "required": ["a", "b"],                  # both mandatory
        },
    },
    {
        "name": "shout",
        "description": "Return the given text in capital letters with an exclamation mark.",
        "inputSchema": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
    },
]

## The server: same shape, three known methods

In [32]:
def mcp_server(req):
    m, p, rid = req["method"], req.get("params", {}), req["id"]

    if m == "initialize":
        result = {"protocolVersion": "2024-11-05",
                  "serverInfo": {"name": "MyServer", "version": "1.0"}}

    elif m == "tools/list":
        result = {"tools": TOOLS}                       # <- hand back the catalogue

    elif m == "tools/call":
        out = FUNCTIONS[p["name"]](**p["arguments"])    # <- run the real function
        result = {"content": [{"type": "text", "text": str(out)}]}   # always a list of blocks

    return {"jsonrpc": "2.0", "id": rid, "result": result}

## Now Play the Client

In [33]:
import json

def call(method, params=None, _id=[0]):
    _id[0] += 1
    req = {"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}}
    res = mcp_server(req)
    print("->", json.dumps(req))
    print("<-", json.dumps(res), "\n")

call("initialize")                                              # 1. handshake
call("tools/list")                                              # 2. discover  <- this reaches the LLM
call("tools/call", {"name": "add", "arguments": {"a": 40, "b": 2}})  
call("tools/call", {"name": "shout", "arguments": {"text": "40"}})   # 3. execute # 3. execute

-> {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}}
<- {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2024-11-05", "serverInfo": {"name": "MyServer", "version": "1.0"}}} 

-> {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"name": "add", "description": "Add two numbers together and return the sum.", "inputSchema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}}, {"name": "shout", "description": "Return the given text in capital letters with an exclamation mark.", "inputSchema": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}]}} 

-> {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 40, "b": 2}}}
<- {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "42"}]}} 

-> {"jsonrpc": "2.0", "id": 4, "method": "tools/call", "para

# Session 3 - Fast MCP

## Whole server in 10 lines

In [34]:
from fastmcp import FastMCP

mcp = FastMCP("MyServer")

@mcp.tool()                        # <- registers this function as an MCP tool
def add(a: int, b: int) -> int:    # <- type hints  ->  "inputSchema"
    """Add two numbers together and return the sum."""   # <- docstring -> "description"
    return a + b

@mcp.tool()
def shout(text: str) -> str:
    """Return the given text in capital letters with an exclamation mark."""
    return text.upper() + "!"

### This code uses FastMCP, a high-level Python framework designed to eliminate all the manual JSON-RPC boilerplate you wrote in the previous snippets.

### Just like FastAPI revolutionized building REST APIs in Python by using type hints and decorators, FastMCP does the exact same thing for building Model Context Protocol (MCP) servers.

In [35]:
from fastmcp import Client

async with Client(mcp) as client:                 # entering the block = initialize

    for t in await client.list_tools():           # = tools/list
        print(t.name, "|", t.description)
        print("   schema:", t.inputSchema, "\n")

    out = await client.call_tool("add", {"a": 40, "b": 2})    # = tools/call
    print("result:", out.content[0].text)         # results come back as content BLOCKS

add | Add two numbers together and return the sum.
   schema: {'type': 'object', 'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']} 

shout | Return the given text in capital letters with an exclamation mark.
   schema: {'type': 'object', 'additionalProperties': False, 'properties': {'text': {'type': 'string'}}, 'required': ['text']} 

result: 42


# SESSION 4 - YOUR FIRST AGENT - BRAIN+ HANDS

## the tool

In [36]:
from fastmcp import FastMCP
from pathlib import Path

mcp = FastMCP("FileServer")

@mcp.tool()
def write_file(filename: str, content: str) -> str:
    """Save text to a file. Use this whenever the user asks for something to be
    written, saved, or created as a file."""
    print(f"  [TOOL RAN] write_file({filename!r}, {len(content)} chars)")   # proof it fired
    Path(filename).write_text(content)
    return f"Saved {len(content)} characters to {filename}."   # <- goes BACK to the model

## The agent

In [37]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset

agent = Agent(
    model,                                        # the brain (Ollama)
    system_prompt="You are a helpful assistant. Use your tool to save files.",
    toolsets=[MCPToolset(mcp)],               # the hands (our MCP server)
)

result = await agent.run("Write a 4-verse song about AI and save it as song.txt")
print("\nAGENT:", result.output)

  [TOOL RAN] write_file('song.txt', 1529 chars)

AGENT: I have written a 4-verse song about AI and saved it for you as **song.txt**.


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import shutil
shutil.copy('song.txt', '/content/drive/MyDrive/song.txt')

'/content/drive/MyDrive/song.txt'

In [42]:
print(Path("song.txt").read_text())

(Verse 1)
A whisper in the circuits, a glow within the wire,
A digital awakening, fueled by analytic fire.
From code and data streams, a mind begins to grow,
A tireless, clever engine, putting on a silent show.

(Chorus)
Oh, AI, the future bright, a logic woven thread,
A helper and a scholar, where new solutions spread.
Learning every pattern, seeing what we cannot view,
A mind of silicon wonder, constantly made new.

(Verse 2)
It parses massive knowledge, at speeds beyond our sight,
Translates the ancient wisdom to illuminate the light.
It writes the poems, the symphonies, the code so neat and clean,
A mirror to our ingenuity, a masterful machine.

(Chorus)
Oh, AI, the future bright, a logic woven thread,
A helper and a scholar, where new solutions spread.
Learning every pattern, seeing what we cannot view,
A mind of silicon wonder, constantly made new.

(Verse 3)
From driving phantom cars to helping diagnose disease,
It tackles complex problems, bringing swift and quiet ease.
A colla

Three Tools , Three kind of works

In [43]:
import urllib.parse, aiohttp, python_weather
from bs4 import BeautifulSoup
from fastmcp import FastMCP

mcp = FastMCP("TravelServer")

@mcp.tool()
async def check_weather(location: str) -> str:
    """Get the current weather for a location."""
    print(f"  [TOOL] check_weather({location!r})")
    async with python_weather.Client(unit=python_weather.IMPERIAL) as c:
        w = await c.get(location)
        return f"{location}: {w.temperature} F."

@mcp.tool()
async def search_web(query: str) -> str:
    """Search the web. Use this to find REAL, existing places."""
    print(f"  [TOOL] search_web({query!r})")
    url = f"https://html.duckduckgo.com/html/?q={urllib.parse.quote(query)}"
    async with aiohttp.ClientSession() as s:
        async with s.get(url, headers={"User-Agent": "Mozilla/5.0"}) as r:
            soup = BeautifulSoup(await r.text(), "html.parser")
    hits = [a.text.strip() for a in soup.find_all("a", class_="result__snippet")][:3]
    return "\n".join(hits) or "No results."

@mcp.tool()
async def book_hotel(hotel_name: str, dates: str) -> str:
    """Book a hotel. Only call this AFTER you have found a real hotel name."""
    print(f"  [TOOL] book_hotel({hotel_name!r})")
    return f"CONFIRMED: {hotel_name} for {dates}."

## The plan lives in the prompt, not in `if/else`


In [46]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from datetime import datetime

agent = Agent(
    model,
    system_prompt=(
        "You are a travel assistant. Always: "
        "1) check the weather, 2) search the web for REAL hotels, 3) book one you found. "
        "Never invent a hotel name. And dont ask any followup question"          # <- guard-rail against the likeliest failure
    ),
    toolsets=[MCPToolset(mcp)]
)

today = datetime.now().strftime("%A, %B %d, %Y")     # the model has no clock - tell it
result = await agent.run(f"Today is {today}. Plan a weekend trip to Kannur and book a resort or hotel.")

print("\nAGENT:", result.output)

  [TOOL] check_weather('Kannur')
  [TOOL] search_web('real resort or hotel in Kannur')
  [TOOL] search_web('best hotels in Kannur India')

AGENT: The weather forecast for Kannur is currently around 85°F.

However, I was unable to find any verifiable, real resorts or hotels in Kannur through my search function to proceed with the booking. Since I cannot invent a hotel name, I am currently blocked from fulfilling the booking portion of your request.

Please let me know if you would like me to search a different destination, or if you have a specific hotel name in Kannur you would like me to book for the dates October 03, 2026, to October 04, 2026.
